# ST-Reason · 02 Detection, tracking and object states

For every video in the study, this notebook:

1. **Decodes** the video once at 5 fps. The same timestamps are reused by notebook 03.
2. **Detects and tracks** people and COCO objects with YOLO26-m + ByteTrack on every decoded frame.
3. **Adds household objects** outside COCO with YOLOE-26 and the fixed vocabulary from the pilot (`config/openvocab.json`), at 1 fps (the pilot showed this is enough and 5× cheaper).
4. **Stitches tracks:** two tracks of the same class are joined when one ends at most 2 s before the other starts and their SigLIP crop embeddings match. The original IDs are kept as well, so E1 can score tracking with and without stitching.
5. **Proposes object states** from box geometry: *held_by*, *on*, *near*, *absent*. These are only candidates. Notebook 03 asks the VLM to confirm each change.

**Two members, in parallel.** Each video belongs to one shard. The split balances video seconds and is computed the same way on every machine. Member A sets `SHARD = 0`, member B sets `SHARD = 1`. Each processes only their own videos, and each video's results are separate files, so the two runs never write the same file.

**Runtime:** GPU (T4). About 1–1.5 hours per member. After a disconnect, run all again and finished videos are skipped. A cell that reaches the time budget (110 min) stops cleanly; run it again to continue.

**How results are saved.** Each video's files are written to the Colab disk first. Every `SYNC_EVERY` videos (and at the end of every run), the new files are zipped into one **pack**, copied to `outputs/cv/` on Drive, and read back from Drive to prove they arrived. Writing thousands of small files straight into a Drive folder can lose them without any error, so this notebook never does. If an upload cannot be verified, the notebook stops with a clear message: then do **not** restart the runtime, because the results are still on the Colab disk. At the start of every run, all packs are unpacked to the Colab disk, so finished videos are known and skipped.

| Inside the packs (`outputs/cv/02_*.zip`) | Content |
| --- | --- |
| `frames_index/<id>.json` | decoded timestamps, size, fps, duration |
| `tracks/<id>.json` | every box per frame with its stitched and original track ID; one summary per track |
| `states/<id>.json` | per-object state intervals and the candidate state changes |

`outputs/cv/work_list.json` (all videos with their shard, written by member A) is a normal file on Drive.

In [53]:
# --- Parameters: the only cell to edit ---------------------------------------------------------
SHARD = 0                 # member A: 0, member B: 1
N_SHARDS = 1
GROUPS = ['dev', 'star', 'main', 'intentqa']   # order of work; 'intentqa' is only needed for Tier 2
LIMIT_PER_GROUP = None    # e.g. 2 for a quick test run; None for the real run
SYNC_EVERY = 25           # upload results to Drive after this many videos (at most this many are redone after a crash)

In [54]:
# --- 0. Bootstrap --------------------------------------------------------------------------
import os, glob
ST_NOTEBOOK = '02_cv_detect_track'
ST_QUIET = True
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')  # before torch loads
if 'COLAB_RELEASE_TAG' in os.environ:
    from google.colab import drive
    drive.mount('/content/drive')
    SETUP_NB = '/content/drive/MyDrive/ST-Reason/notebooks/00_setup_and_utils.ipynb'
elif os.path.exists('/kaggle/working'):
    SETUP_NB = (glob.glob('/kaggle/input/**/00_setup_and_utils.ipynb', recursive=True) or ['00_setup_and_utils.ipynb'])[0]
else:
    SETUP_NB = os.environ.get('ST_SETUP_NB', '00_setup_and_utils.ipynb')
%run -i $SETUP_NB

MOCK = os.environ.get('ST_MOCK_MODELS') == '1'      # local testing only: fake models, no GPU
assert SHARD in range(N_SHARDS), 'SHARD must be 0 .. N_SHARDS-1'
# each member appends to their own manifest, so two people never write the same file at once
MANIFEST_PATH = P['runs'] / f'manifest_shard{SHARD}.csv'
# Log to the Colab disk; a copy goes to Drive with every upload. (Appending to a file on Drive line by line
# makes Drive store many duplicate copies of it.)
DRIVE_LOG_PATH = LOG_PATH
LOG_PATH = P['local'] / 'logs' / LOG_PATH.name
LOG_PATH.parent.mkdir(parents=True, exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Runtime: colab | project folder: /content/drive/MyDrive/ST-Reason | local scratch: /content/st_local
config hash: 2a9e0e4f87


## 1. Work list and shards

The videos come from four groups: the dev set, the STAR clips (original versions only), the main-grid questions actually used (`main_grid_eval.csv`, 400 questions after the pilot rule), and the IntentQA evaluation set (Tier 2, last).

`main_grid_eval.csv` is created here the first time, by member A, and is frozen afterwards. It contains all 200 robustness questions plus a type-stratified sample of the rest of the main grid. The pause and sham STAR versions are not processed here; notebook 03 derives them from the original clips by shifting times (see 03).

In [55]:
# --- Shared CV helpers (this cell is identical in 02 and 03) -------------------------------------
import pandas as pd, numpy as np, cv2
from PIL import Image, ImageDraw, ImageFont

CVC = CFG['cv']
CVC.setdefault('openvocab_fps', 1.0)          # pilot: YOLOE at 1 fps is enough and 5x cheaper
CVC.setdefault('stitch_min_cos', 0.80)
CVC.setdefault('state_fill_gap_s', 1.0)
CVC.setdefault('state_min_s', 0.6)
CVC.setdefault('event_window_s', CVC['window_s'])   # pilot: non-overlapping 3 s windows
# Per-video files are written to the fast Colab disk (LOCAL_CV) and uploaded to Drive in batches, as a
# few zip "packs" placed directly in outputs/cv/. Each pack is read back from Drive after upload to prove
# it arrived. (Writing thousands of small files to Drive can silently lose them, and new sub-folders made
# from Colab inside a folder shared from someone else's Drive can vanish, so none are created.)
LOCAL_CV = P['local'] / 'cv_out'
PACKS = P['cv']
PACK_GLOB = '0[23]*_shard*.zip'
CVD = {k: LOCAL_CV / k for k in ['frames_index', 'tracks', 'states', 'events_raw', 'events', 'captions', 'frames']}
for _d in CVD.values():
    _d.mkdir(parents=True, exist_ok=True)
_PACKED = set()            # paths (relative to LOCAL_CV) already safely inside a pack on Drive

def restore_from_packs(prefixes=None):
    """Unpack every pack on Drive into LOCAL_CV (files already there are skipped).
    prefixes: only these top-level folders, e.g. ['states', 'tracks']."""
    n_new, bad = 0, []
    for z in sorted(PACKS.glob(PACK_GLOB)):
        try:
            with zipfile.ZipFile(z) as zf:
                for m in zf.namelist():
                    _PACKED.add(m)
                    if prefixes and m.split('/')[0] not in prefixes:
                        continue
                    out = LOCAL_CV / m
                    if not out.exists():
                        out.parent.mkdir(parents=True, exist_ok=True)
                        with zf.open(m) as s, open(out, 'wb') as d:
                            shutil.copyfileobj(s, d)
                        n_new += 1
        except (zipfile.BadZipFile, OSError) as e:
            bad.append(z.name)
    if bad:
        log(f'{len(bad)} damaged packs skipped (an upload was cut off; their videos are redone): {bad[:3]}', 'WARN')
    log(f'restored {n_new} files from {len(list(PACKS.glob(PACK_GLOB)))} packs on Drive')

class DriveSyncError(BaseException):
    """Stops the whole loop (run_resumable only skips ordinary errors), so no more work piles up unsaved."""

def sync_to_drive(subdirs, tag):
    """Upload every local file in `subdirs` that is not yet on Drive, as one new pack, and verify it."""
    new = []
    for sd in subdirs:
        for p in (LOCAL_CV / sd).rglob('*'):
            if p.is_file() and not p.name.endswith('.tmp'):
                rel = str(p.relative_to(LOCAL_CV))
                if rel not in _PACKED:
                    new.append(rel)
    if not new:
        return 0
    name = f'{tag}_shard{SHARD}_{dt.datetime.now():%Y%m%d-%H%M%S}_{len(new):05d}.zip'
    tmp = P['local_tmp'] / name
    with zipfile.ZipFile(tmp, 'w', zipfile.ZIP_DEFLATED, compresslevel=6) as zf:
        for rel in new:
            zf.write(LOCAL_CV / rel, rel, compress_type=zipfile.ZIP_STORED if rel.endswith('.jpg') else zipfile.ZIP_DEFLATED)
    dest = PACKS / name
    for attempt in range(3):
        try:
            shutil.copyfile(tmp, dest)
            with zipfile.ZipFile(dest) as zf:                  # read back from Drive
                ok = len(zf.namelist()) == len(new) and zf.testzip() is None
            if ok:
                break
        except Exception as e:
            ok = False
            log(f'upload of {name} failed ({e.__class__.__name__}: {e}); retrying', 'WARN')
        time.sleep(5 * (attempt + 1))
    if not ok:
        raise DriveSyncError(f'Drive did not keep {name}. The results are still on this Colab disk: do NOT restart '
                           'the runtime. Check Drive space and that you can edit MyDrive/ST-Reason, then run the '
                           'sync cell again.')
    _PACKED.update(new)
    tmp.unlink(missing_ok=True)
    try:
        shutil.copyfile(LOG_PATH, DRIVE_LOG_PATH)          # one overwrite per upload, not one per line
    except Exception:
        pass
    log(f'uploaded {name}: {len(new)} files ({_gb(dest.stat().st_size) * 1024:.1f} MB), verified on Drive')
    return len(new)

restore_from_packs()

COCO = set(['person', 'bicycle', 'car', 'motorcycle', 'airplane', 'bus', 'train', 'truck', 'boat', 'traffic light',
    'fire hydrant', 'stop sign', 'parking meter', 'bench', 'bird', 'cat', 'dog', 'horse', 'sheep', 'cow', 'elephant',
    'bear', 'zebra', 'giraffe', 'backpack', 'umbrella', 'handbag', 'tie', 'suitcase', 'frisbee', 'skis', 'snowboard',
    'sports ball', 'kite', 'baseball bat', 'baseball glove', 'skateboard', 'surfboard', 'tennis racket', 'bottle',
    'wine glass', 'cup', 'fork', 'knife', 'spoon', 'bowl', 'banana', 'apple', 'sandwich', 'orange', 'broccoli',
    'carrot', 'hot dog', 'pizza', 'donut', 'cake', 'chair', 'couch', 'potted plant', 'bed', 'dining table', 'toilet',
    'tv', 'laptop', 'mouse', 'remote', 'keyboard', 'cell phone', 'microwave', 'oven', 'toaster', 'sink',
    'refrigerator', 'book', 'clock', 'vase', 'scissors', 'teddy bear', 'hair drier', 'toothbrush'])
# objects that other objects can rest "on"
SURFACES = {'dining table', 'table', 'desk', 'bed', 'couch', 'sofa', 'chair', 'bench', 'shelf', 'counter',
            'cabinet', 'toilet', 'sink', 'mat', 'rug', 'carpet', 'tray', 'stool', 'cushion', 'pillow'}

def save_compact(obj, path):
    """Atomic JSON write without indentation (CV files are large)."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + '.tmp')
    with open(tmp, 'w') as f:
        json.dump(obj, f, separators=(',', ':'), ensure_ascii=False, default=_json_default)
        f.flush(); os.fsync(f.fileno())
    os.replace(tmp, path)

def decode_frames(path, fps_out):
    """Yield (t_seconds, BGR frame) at about fps_out, decoding once and streaming.
    02 and 03 both use this, so every stage sees exactly the same timestamps."""
    cap = cv2.VideoCapture(str(path))
    src_fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    step, i, next_t = 1.0 / fps_out, 0, 0.0
    while True:
        ok, fr = cap.read()
        if not ok:
            break
        t = i / src_fps
        if t + 1e-6 >= next_t:
            yield round(t, 3), fr
            next_t += step
        i += 1
    cap.release()

def stratified_subset(df, n, seed):
    """Exactly n rows, allocated to question types in proportion to their counts (same as 01)."""
    n = min(n, len(df))
    counts = df.type.value_counts()
    alloc = (counts / counts.sum() * n).round().astype(int).clip(upper=counts)
    order, i = list(counts.index), 0
    while alloc.sum() != n:
        t = order[i % len(order)]
        if alloc.sum() < n and alloc[t] < counts[t]:
            alloc[t] += 1
        elif alloc.sum() > n and alloc[t] > 0:
            alloc[t] -= 1
        i += 1
    parts = [df[df.type == t].sample(n=int(k), random_state=seed) for t, k in alloc.items() if k > 0]
    return pd.concat(parts).sample(frac=1, random_state=seed).reset_index(drop=True)

def main_grid_eval():
    """The questions the main grid is actually run on. The pilot rule cut the grid to
    config data.main_grid_n (400). The subset keeps all 200 robustness questions and adds a
    stratified sample of the rest, so the robustness subset stays inside it. Frozen once saved."""
    path = P['subsets'] / 'main_grid_eval.csv'
    if path.exists():
        return pd.read_csv(path, dtype={'video_id': str})
    grid = pd.read_csv(P['subsets'] / 'main_grid.csv', dtype={'video_id': str})
    n = int(CFG['data'].get('main_grid_n') or (CFG.get('pilot_decisions') or {}).get('main_grid_n') or len(grid))
    if n >= len(grid):
        sub = grid
    else:
        rob = pd.read_csv(P['subsets'] / 'robustness.csv', dtype={'video_id': str})
        if n <= len(rob):
            keep = set(stratified_subset(rob, n, CFG['seed'] + 5).uid)
        else:
            rest = grid[~grid.uid.isin(set(rob.uid))]
            keep = set(rob.uid) | set(stratified_subset(rest, n - len(rob), CFG['seed'] + 5).uid)
        sub = grid[grid.uid.isin(keep)].reset_index(drop=True)
    if SHARD == 0:                                  # only one member writes shared files
        sub.to_csv(path, index=False)
        hp = P['subsets'] / 'set_hashes.json'
        hashes = load_json(hp, default={})
        hashes['main_grid_eval'] = hashlib.sha1(sub.uid.astype(str).str.cat(sep='|').encode()).hexdigest()[:12]
        save_json(hashes, hp)
        log(f'main_grid_eval.csv frozen: {len(sub)} questions, {sub.video_id.nunique()} videos')
    return sub

_ZIP_INDEX = {}
def zip_index(name):
    """{video stem: member path} for a video zip on Drive (cached)."""
    if name not in _ZIP_INDEX:
        zp = P['data'] / name
        if not zp.exists():
            _ZIP_INDEX[name] = {}
        else:
            with zipfile.ZipFile(zp) as zf:
                _ZIP_INDEX[name] = {Path(n).stem: n for n in zf.namelist() if n.lower().endswith('.mp4')}
    return _ZIP_INDEX[name]

GROUP_ORDER = {'dev': 0, 'star': 1, 'main': 2, 'intentqa': 3}
def build_work_list():
    """Every video the CV pipeline must process, with its source zip, duration and shard.
    Shards are balanced by total video seconds, tier by tier, and are the same on every
    member's machine (deterministic), so nobody needs to coordinate."""
    meta = pd.read_csv(P['ann'] / 'video_meta.csv', dtype={'video_id': str}).set_index('video_id').duration_s
    rows, missing = [], {}
    nextqa_sets = [('dev', pd.read_csv(P['subsets'] / 'dev.csv', dtype={'video_id': str}), 'videos_nextqa_eval.zip', 1),
                   ('main', main_grid_eval(), 'videos_nextqa_eval.zip', 1),
                   ('intentqa', pd.read_csv(P['subsets'] / 'intentqa_eval.csv', dtype={'video_id': str}),
                    'videos_intentqa_eval.zip', 2)]
    for group, df, zname, tier in nextqa_sets:
        idx = zip_index(zname)
        for v in sorted(df.video_id.unique()):
            if v not in idx:
                missing.setdefault(group, []).append(v); continue
            rows.append({'key': v, 'group': group, 'tier': tier, 'zip': zname, 'member': idx[v],
                         'duration_s': meta.get(v, np.nan)})
    man = P['data'] / 'star' / 'clip_manifest.csv'
    if man.exists():
        M = pd.read_csv(man)
        idx = zip_index('videos_star_eval.zip')
        for r in M[M.version == 'orig'].itertuples(index=False):
            if r.clip_id not in idx:
                missing.setdefault('star', []).append(r.clip_id); continue
            rows.append({'key': r.clip_id, 'group': 'star', 'tier': 1, 'zip': 'videos_star_eval.zip',
                         'member': idx[r.clip_id], 'duration_s': r.n_frames_src / r.fps})
    else:
        log('data/star/clip_manifest.csv not found: run 01b_star first. STAR clips are skipped for now.', 'WARN')
    W = pd.DataFrame(rows).drop_duplicates('key').reset_index(drop=True)
    W['duration_s'] = W.duration_s.fillna(W.duration_s.median()).astype(float)
    W['shard'] = -1
    for tier in sorted(W.tier.unique()):              # greedy: longest video to the least-loaded shard
        load = [0.0] * N_SHARDS
        sub = W[W.tier == tier].sort_values(['duration_s', 'key'], ascending=[False, True])
        for i, d in zip(sub.index, sub.duration_s):
            s = int(np.argmin(load)); W.at[i, 'shard'] = s; load[s] += d
    W['order'] = W.group.map(GROUP_ORDER)
    W = W.sort_values(['tier', 'order', 'key']).drop(columns='order').reset_index(drop=True)
    for g, v in missing.items():
        log(f'{len(v)} {g} videos are not in their zip and are skipped: {v[:5]}', 'WARN')
    return W

class VideoSource:
    """Extracts one video at a time from its Drive zip to the Colab disk, and deletes it after use."""
    def __init__(self):
        self.dir = P['local_videos'] / 'cv'
        self.dir.mkdir(parents=True, exist_ok=True)
        self.zf = {}
    def get(self, row):
        out = self.dir / f'{row.key}.mp4'
        if not out.exists():
            if row.zip not in self.zf:
                self.zf[row.zip] = zipfile.ZipFile(P['data'] / row.zip)
            part = out.with_suffix('.part')
            with self.zf[row.zip].open(row.member) as s, open(part, 'wb') as d:
                shutil.copyfileobj(s, d)
            os.replace(part, out)
        return out
    def drop(self, row):
        (self.dir / f'{row.key}.mp4').unlink(missing_ok=True)

def select_work(W, groups, limit=None):
    """This member's videos, in processing order (Tier 1 first)."""
    mine = W[(W.shard == SHARD) & W.group.isin(groups)]
    if limit:
        mine = mine.groupby('group', sort=False).head(limit)
    mine = mine.assign(_o=mine.group.map({g: i for i, g in enumerate(groups)}))
    return mine.sort_values(['tier', '_o', 'key']).drop(columns='_o').reset_index(drop=True)

def delete_model_cache(*repo_ids):
    """Free Colab disk: remove downloaded Hugging Face weights that are no longer needed."""
    hub = Path(os.environ.get('HF_HOME', Path.home() / '.cache' / 'huggingface')) / 'hub'
    for rid in repo_ids:
        d = hub / ('models--' + rid.replace('/', '--'))
        if d.exists():
            shutil.rmtree(d, ignore_errors=True)
            log(f'deleted cached weights: {rid}')

WORK = build_work_list()
if SHARD == 0:
    save_compact(WORK.to_dict('records'), P['cv'] / 'work_list.json')
summary = WORK.pivot_table(index='group', columns='shard', values='duration_s', aggfunc=['count', 'sum'])
summary.columns = [f'{a}_shard{b}' for a, b in summary.columns]
print(summary.round(0).astype(int).to_string())

[14:13:15] INFO  restored 0 files from 1 packs on Drive
[14:13:15] INFO  main_grid_eval.csv frozen: 400 questions, 175 videos
          count_shard0  sum_shard0
group                             
dev                 92        3880
intentqa           224       10612
main               175        6898
star               360        4753


In [56]:
MY = select_work(WORK, GROUPS, LIMIT_PER_GROUP)
done = done_ids(CVD['states'])
print(f'shard {SHARD}: {len(MY)} videos ({MY.duration_s.sum() / 60:.0f} min of video), '
      f'{len(set(MY.key) & done)} already done')
print(MY.groupby('group', sort=False).agg(videos=('key', 'size'), minutes=('duration_s', lambda s: round(s.sum() / 60, 1))))

shard 0: 851 videos (436 min of video), 47 already done
          videos  minutes
group                    
dev           92     64.7
star         360     79.2
main         175    115.0
intentqa     224    176.9


## 2. Models

- **YOLO26-m** (detection) with **ByteTrack**. Detections down to confidence 0.1 are passed to the tracker, which uses low-confidence boxes to keep existing tracks alive. Untracked boxes are kept only at confidence ≥ 0.25. The tracker is reset at the start of every video.
- **YOLOE-26** with the fixed vocabulary (COCO-80 plus the open-vocabulary nouns). Only its non-COCO detections are kept, because YOLO26 already covers COCO. These are linked into tracks by box overlap.
- **SigLIP** crop embeddings for stitching. If SigLIP cannot be loaded, a colour-histogram embedding is used instead, with a stricter threshold, and this is recorded in every output file.

In [57]:
if not MOCK:
    # lap (ByteTrack) and CLIP (YOLOE text prompts) are installed up front, so nothing auto-installs mid-run
    pip_install('ultralytics>=8.4.0', 'lap>=0.5.12', 'git+https://github.com/ultralytics/CLIP.git')
try:
    import torch
    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
except ImportError:
    if not MOCK:
        raise
    DEVICE = 'cpu'
if DEVICE == 'cpu' and not MOCK:
    log('No GPU: this works but is slow. Switch to a T4 runtime for the real run.', 'WARN')

OPENVOCAB = load_json(P['config'] / 'openvocab.json', default={}).get('nouns', [])
assert OPENVOCAB or MOCK, 'config/openvocab.json is missing: it is made by 00b_pilot section 5'
TRACK_CONF = min(0.1, CVC['det_conf'])

def load_yolo(names):
    from ultralytics import YOLO
    last = None
    for n in names:
        try:
            return YOLO(n), n
        except Exception as e:
            last = e
    raise last

if MOCK:
    DET_NAME, OV_NAME = 'mock', 'mock'
else:
    DET, DET_NAME = load_yolo([CVC['det_model'], 'yolo26m.pt', 'yolo11m.pt'])
    OV, OV_NAME = load_yolo([CVC['openvocab_model'], 'yoloe-26m-seg.pt', 'yoloe-11m-seg.pt'])
    OV_CLASSES = sorted(set(OPENVOCAB) | COCO)
    try:
        OV.set_classes(OV_CLASSES, OV.get_text_pe(OV_CLASSES))
    except (TypeError, AttributeError):
        OV.set_classes(OV_CLASSES)

# crop embedder for stitching
EMB = {'method': None}
if not MOCK:
    try:
        from transformers import AutoModel, AutoImageProcessor
        spec = CFG['models']['crop_embedder']
        EMB['proc'] = AutoImageProcessor.from_pretrained(spec['id'], revision=spec.get('revision', 'main'))
        EMB['model'] = AutoModel.from_pretrained(spec['id'], revision=spec.get('revision', 'main')).vision_model.to(DEVICE).eval()
        EMB.update(method='siglip', threshold=CVC['stitch_min_cos'])
    except Exception as e:
        log(f'SigLIP could not be loaded ({e.__class__.__name__}): stitching uses colour histograms', 'WARN')
if EMB['method'] is None:
    EMB.update(method='hsv_hist', threshold=CVC.get('stitch_min_cos_hist', 0.90))

def embed_crops(crops):
    """L2-normalised embeddings for a list of RGB uint8 arrays."""
    if not crops:
        return np.zeros((0, 1), np.float32)
    if EMB['method'] == 'siglip':
        out = []
        for i in range(0, len(crops), 64):
            px = EMB['proc'](images=[Image.fromarray(c) for c in crops[i:i + 64]], return_tensors='pt')['pixel_values']
            with torch.inference_mode():
                out.append(EMB['model'](pixel_values=px.to(DEVICE)).pooler_output.float().cpu().numpy())
        v = np.concatenate(out)
    else:
        v = np.stack([cv2.calcHist([cv2.cvtColor(c, cv2.COLOR_RGB2HSV)], [0, 1, 2], None, [8, 4, 4],
                                   [0, 180, 0, 256, 0, 256]).ravel() for c in crops]).astype(np.float32)
    return v / (np.linalg.norm(v, axis=1, keepdims=True) + 1e-8)
print(f'detector {DET_NAME}, open vocabulary {OV_NAME} ({len(OPENVOCAB)} extra nouns), stitching by {EMB["method"]}')

[14:13:21] INFO  pip install ultralytics>=8.4.0 lap>=0.5.12 git+https://github.com/ultralytics/CLIP.git


Loading weights:   0%|          | 0/408 [00:00<?, ?it/s]

detector yolo26m.pt, open vocabulary yoloe-26m-seg.pt (100 extra nouns), stitching by siglip


## 3. Per-frame detection and tracking

For each video: reset the tracker, stream frames at 5 fps, track every frame and run YOLOE every 5th frame (1 fps). For each track, the crops at its first and last frames are kept for stitching. Nothing else is held in memory.

In [58]:
def reset_tracker():
    pr = getattr(DET, 'predictor', None)
    for tr in (getattr(pr, 'trackers', None) or []):
        tr.reset()

def detect_track(fr):
    """[(track_id or None, class, xyxy, conf)] for one frame."""
    r = DET.track(fr, persist=True, tracker=CVC['tracker'], conf=TRACK_CONF, verbose=False)[0]
    b = r.boxes
    if b is None or len(b) == 0:
        return []
    ids = b.id.int().tolist() if b.id is not None else [None] * len(b)
    out = []
    for i, c, xyxy, cf in zip(ids, b.cls.tolist(), b.xyxy.tolist(), b.conf.tolist()):
        if i is None and cf < CVC['det_conf']:
            continue                                  # low-confidence boxes only count inside tracks
        out.append((i, r.names[int(c)], [round(float(x), 1) for x in xyxy], round(float(cf), 3)))
    return out

def detect_openvocab(fr):
    r = OV.predict(fr, conf=CVC['det_conf'], verbose=False)[0]
    b = r.boxes
    if b is None or len(b) == 0:
        return []
    return [(r.names[int(c)], [round(float(x), 1) for x in xyxy], round(float(cf), 3))
            for c, xyxy, cf in zip(b.cls.tolist(), b.xyxy.tolist(), b.conf.tolist()) if r.names[int(c)] not in COCO]

if MOCK:   # scripted scene: a person, a table, a cup that is put down, picked up, carried and put back,
           # with one ID switch (the cup's track breaks for 0.6 s), a toy found only by the open vocabulary,
           # and glasses that are worn (no state)
    def reset_tracker():
        pass
    def detect_track(fr, _t=[0.0]):
        h, w = fr.shape[:2]; t = MOCK_T[0]
        S = lambda x1, y1, x2, y2: [round(x1 * w, 1), round(y1 * h, 1), round(x2 * w, 1), round(y2 * h, 1)]
        out = [(1, 'person', S(.15, .10, .45, .95), .9), (2, 'dining table', S(.05, .60, .95, .95), .8)]
        if t < 3:
            out.append((3, 'cup', S(.60, .52, .66, .60), .7))          # on the table
        elif t < 4.6:
            out.append((3, 'cup', S(.30, .40, .36, .48), .6))          # in the person's hands
        elif 5.2 <= t < 7:
            out.append((9, 'cup', S(.31, .41, .37, .49), .6))          # same cup, new ID after a gap
        elif t >= 7:
            out.append((9, 'cup', S(.50, .52, .56, .60), .7))          # put back on the table
        return out
    def detect_openvocab(fr):
        h, w = fr.shape[:2]
        return ([('toy', [0.70 * w, 0.85 * h, 0.80 * w, 0.95 * h], 0.4)] if MOCK_T[0] > 2 else []) + \
               [('glasses', [0.25 * w, 0.15 * h, 0.33 * w, 0.20 * h], 0.5)]
MOCK_T = [0.0]

def crop(fr, xyxy, size=112):
    h, w = fr.shape[:2]
    x1, y1, x2, y2 = [int(round(v)) for v in xyxy]
    x1, y1, x2, y2 = max(0, x1), max(0, y1), min(w, x2), min(h, y2)
    if x2 - x1 < 2 or y2 - y1 < 2:
        return None
    return cv2.resize(cv2.cvtColor(fr[y1:y2, x1:x2], cv2.COLOR_BGR2RGB), (size, size))

def iou(a, b):
    ix = max(0, min(a[2], b[2]) - max(a[0], b[0])); iy = max(0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = ix * iy
    u = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / u if u > 0 else 0.0

def link_openvocab(ov_frames, max_gap_s):
    """Greedy same-class linking of the 1 fps YOLOE boxes into tracks. At 1 fps a small object can
    move further than its own size, so boxes link by overlap (IoU >= 0.3) or, failing that, when the
    centres are within 1.5 box sizes."""
    tracks, out, n = [], {}, 0
    def dist(a, b):
        ca, cb = ((a[0] + a[2]) / 2, (a[1] + a[3]) / 2), ((b[0] + b[2]) / 2, (b[1] + b[3]) / 2)
        size = max(a[2] - a[0], a[3] - a[1], b[2] - b[0], b[3] - b[1], 1.0)
        return ((ca[0] - cb[0]) ** 2 + (ca[1] - cb[1]) ** 2) ** 0.5 / size
    for t, boxes in ov_frames:
        used = set()
        for k, (cls, xyxy, cf) in sorted(enumerate(boxes), key=lambda kv: -kv[1][2]):
            best, best_score = None, None
            for tr in tracks:
                if tr['cls'] != cls or not (0 < t - tr['t'] <= max_gap_s) or id(tr) in used:
                    continue
                v, d = iou(tr['box'], xyxy), dist(tr['box'], xyxy)
                score = (0, -v) if v >= 0.3 else ((1, d) if d <= 1.5 else None)
                if score is not None and (best_score is None or score < best_score):
                    best, best_score = tr, score
            if best is None:
                n += 1; best = {'cls': cls, 'id': f'{cls.replace(" ", "_")}_ov{n}'}; tracks.append(best)
            best.update(t=t, box=xyxy); used.add(id(best))
            out[(t, k)] = best['id']
    return out

def scan_video(path):
    """One pass over the video: per-frame tracked boxes, 1 fps open-vocabulary boxes, edge crops."""
    reset_tracker()
    fps_out = CVC['decode_fps']
    ov_every = max(1, int(round(fps_out / CVC['openvocab_fps'])))
    frames, ov_frames, first_crop, last_crop = [], [], {}, {}
    for i, (t, fr) in enumerate(decode_frames(path, fps_out)):
        MOCK_T[0] = t
        dets = detect_track(fr)
        frames.append({'t': t, 'det': dets})
        for tid, cls, xyxy, cf in dets:
            if tid is None:
                continue
            c = crop(fr, xyxy)
            if c is not None:
                first_crop.setdefault(tid, c); last_crop[tid] = c
        if i % ov_every == 0:
            ov_frames.append((t, detect_openvocab(fr)))
        if i == 0:
            shape = fr.shape
    if not frames:
        raise RuntimeError('no frames could be decoded')
    return frames, ov_frames, first_crop, last_crop, shape

## 4. Track IDs and stitching

- Each track gets the class it has in most of its frames, and an ID such as `cup_3`. Every box keeps its tracker ID as `raw`.
- **Stitching:** for each pair of same-class tracks where B starts 0–2 s after A ends, the cosine similarity of A's last crop and B's first crop is computed. Pairs above the threshold are joined, closest in time first, and each track joins at most one before and one after. Joined tracks take the ID of the first track in the chain.
- Tracks seen for under about 0.6 s (fewer than 3 frames at 5 fps, or a single 1 fps open-vocabulary sighting) are kept and flagged `short`.

In [59]:
from collections import Counter

def build_tracks(frames, ov_frames, first_crop, last_crop):
    maj = {}
    for f in frames:
        for tid, cls, _, _ in f['det']:
            if tid is not None:
                maj.setdefault(tid, Counter())[cls] += 1
    raw_name = {tid: f'{c.most_common(1)[0][0].replace(" ", "_")}_{tid}' for tid, c in maj.items()}
    raw_cls = {tid: c.most_common(1)[0][0] for tid, c in maj.items()}
    span = {}
    for f in frames:
        for tid, *_ in f['det']:
            if tid is not None:
                s = span.setdefault(tid, [f['t'], f['t'], 0]); s[1] = f['t']; s[2] += 1

    # stitching
    ids = [t for t in span if t in first_crop and t in last_crop]
    E_first = dict(zip(ids, embed_crops([first_crop[t] for t in ids])))
    E_last = dict(zip(ids, embed_crops([last_crop[t] for t in ids])))
    cands = []
    for a in ids:
        for b in ids:
            gap = span[b][0] - span[a][1]
            if a != b and raw_cls[a] == raw_cls[b] and 0 < gap <= CVC['stitch_max_gap_s'] + 1e-6:
                sim = float(E_last[a] @ E_first[b])
                if sim >= EMB['threshold']:
                    cands.append((gap, -sim, a, b))
    nxt, prv, links = {}, {}, []
    for gap, nsim, a, b in sorted(cands):
        if a not in nxt and b not in prv:
            nxt[a], prv[b] = b, a
            links.append({'from': raw_name[a], 'to': raw_name[b], 'gap_s': round(gap, 2), 'cos': round(-nsim, 3)})
    root = {}
    for t in span:
        if t not in prv:
            r, cur = t, t
            while cur is not None:
                root[cur] = r; cur = nxt.get(cur)
    sid = {t: raw_name[root.get(t, t)] for t in span}

    # open-vocabulary track IDs
    ov_id = link_openvocab(ov_frames, CVC['stitch_max_gap_s'] + 1.0 / CVC['openvocab_fps'])
    ov_at = {t: boxes for t, boxes in ov_frames}

    out_frames, summ = [], {}
    def add(tr_id, cls, t, src, raw):
        s = summ.setdefault(tr_id, {'cls': cls, 't0': t, 't1': t, 'frames': 0, 'src': src, 'raw_ids': []})
        s['t1'] = t; s['frames'] += 1
        if raw not in s['raw_ids']:
            s['raw_ids'].append(raw)
    for f in frames:
        boxes = []
        for tid, cls, xyxy, cf in f['det']:
            if tid is None:
                boxes.append({'id': None, 'raw': None, 'cls': cls, 'xyxy': xyxy, 'conf': cf, 'src': 'det'})
            else:
                boxes.append({'id': sid[tid], 'raw': raw_name[tid], 'cls': raw_cls[tid], 'xyxy': xyxy, 'conf': cf, 'src': 'det'})
                add(sid[tid], raw_cls[tid], f['t'], 'det', raw_name[tid])
        for k, (cls, xyxy, cf) in enumerate(ov_at.get(f['t'], [])):
            oid = ov_id[(f['t'], k)]
            boxes.append({'id': oid, 'raw': oid, 'cls': cls, 'xyxy': [round(float(x), 1) for x in xyxy], 'conf': cf, 'src': 'ov'})
            add(oid, cls, f['t'], 'ov', oid)
        out_frames.append({'t': f['t'], 'boxes': boxes})
    for s in summ.values():   # short = seen for under about 0.6 s (3 frames at 5 fps, 1 frame at 1 fps)
        s['short'] = s['frames'] < (CVC['min_track_frames_flag'] if s['src'] == 'det' else 2)
    return out_frames, summ, links

## 5. Candidate object states

For every tracked object other than people, each frame gets one relation, checked in this order:

| State | Rule (boxes in the same frame) |
| --- | --- |
| `held_by person_k` | ≥ 60% of the object's box lies inside the person's box, and the object is less than half the person's size |
| `on surface_j` | the object's bottom edge is at the surface's top (within 15% of the object's height above, 60% of the surface's height below), with ≥ 50% horizontal overlap |
| `near person_k` | the object touches the person's box enlarged by 25% on each side (not used for furniture) |
| `visible` | none of the above |
| `absent` | the object is not seen for more than 1 s (also before its first and after its last appearance) |

Worn items (glasses, shoes, hats, clothing, …) and tracks flagged `short` get no states: worn items sit inside a person's box all the time, and very short tracks are mostly detector noise. Both stay in the tracks file.

Labels are smoothed over 3 consecutive observations, runs shorter than 0.6 s are absorbed into the previous run, and gaps of up to 1 s are bridged. Every change from one interval to the next becomes a **candidate change** (`C1`, `C2`, …) for the VLM to confirm in notebook 03. *inside* is not used, because boxes cannot show it.

In [60]:
def _area(b):
    return max(0.0, b[2] - b[0]) * max(0.0, b[3] - b[1])

def _inter(a, b):
    return max(0.0, min(a[2], b[2]) - max(a[0], b[0])) * max(0.0, min(a[3], b[3]) - max(a[1], b[1]))

def relation(ob, persons, surfaces, allow_near=True):
    """(state, target) for one object box in one frame. Furniture is never 'near' a person
    (it would flip on and off whenever someone walks past)."""
    a = _area(ob)
    best = None
    for pid, pb in persons:
        f = _inter(ob, pb) / a if a else 0
        if f >= 0.6 and a < 0.5 * _area(pb) and (best is None or f > best[0]):
            best = (f, pid)
    if best:
        return ('held_by', best[1])
    best = None
    ow, oh = ob[2] - ob[0], ob[3] - ob[1]
    for sid_, sb in surfaces:
        hov = max(0.0, min(ob[2], sb[2]) - max(ob[0], sb[0])) / ow if ow > 0 else 0
        top, sh = sb[1], sb[3] - sb[1]
        if hov >= 0.5 and top - 0.15 * oh <= ob[3] <= top + 0.6 * sh and a < _area(sb) and (best is None or hov > best[0]):
            best = (hov, sid_)
    if best:
        return ('on', best[1])
    if not allow_near:
        return ('visible', None)
    best = None
    cx, cy = (ob[0] + ob[2]) / 2, (ob[1] + ob[3]) / 2
    for pid, pb in persons:
        mx, my = 0.25 * (pb[2] - pb[0]), 0.25 * (pb[3] - pb[1])
        if a < _area(pb) and _inter(ob, [pb[0] - mx, pb[1] - my, pb[2] + mx, pb[3] + my]) > 0:
            d = ((cx - (pb[0] + pb[2]) / 2) ** 2 + (cy - (pb[1] + pb[3]) / 2) ** 2) ** 0.5
            if best is None or d < best[0]:
                best = (d, pid)
    if best:
        return ('near', best[1])
    return ('visible', None)

# worn items sit inside a person's box all the time; "held_by" would be wrong and they never "move"
WEARABLES = {'glasses', 'sunglasses', 'hat', 'cap', 'helmet', 'shoe', 'shoes', 'sock', 'socks', 'shirt', 'jacket',
             'coat', 'dress', 'scarf', 'glove', 'gloves', 'ring', 'necklace', 'bracelet', 'watch', 'tie', 'mask',
             'backpack', 'costume', 'diaper', 'bib', 'pants', 'shorts', 'skirt', 'boot', 'boots'}

def build_states(frames, summ, duration):
    fill, min_s = CVC['state_fill_gap_s'], CVC['state_min_s']
    obs = {}                                   # object id -> [(t, (state, target))]
    for f in frames:
        persons = [(b['id'], b['xyxy']) for b in f['boxes'] if b['id'] and b['cls'] == 'person']
        surfaces = [(b['id'], b['xyxy']) for b in f['boxes'] if b['id'] and b['cls'] in SURFACES]
        for b in f['boxes']:
            if b['id'] and b['cls'] != 'person' and b['cls'] not in WEARABLES and not summ[b['id']]['short']:
                obs.setdefault(b['id'], []).append(
                    (f['t'], relation(b['xyxy'], persons, [s for s in surfaces if s[0] != b['id']],
                                      allow_near=b['cls'] not in SURFACES)))
    objects, changes = {}, []
    for oid, seq in obs.items():
        step = 1.0 / (CVC['openvocab_fps'] if summ[oid]['src'] == 'ov' else CVC['decode_fps'])
        labels = [lab for _, lab in seq]
        if len(labels) >= 3:                   # majority filter over 3 observations
            sm = labels[:1]
            for i in range(1, len(labels) - 1):
                trio = labels[i - 1:i + 2]
                sm.append(max(set(trio), key=trio.count) if trio.count(labels[i]) < 2 else labels[i])
            labels = sm + labels[-1:]
        gap_max = max(fill, 2 * step) + 1e-6       # bridges one missed sighting at 1 fps
        runs = []                              # [state, target, t0, t1]
        prev_t = None
        for (t, _), lab in zip(seq, labels):
            if prev_t is not None and t - prev_t > gap_max:
                runs.append(['absent', None, prev_t + step, t])
            if runs and runs[-1][0] == lab[0] and runs[-1][1] == lab[1] and t - runs[-1][3] <= gap_max:
                runs[-1][3] = t + step
            else:
                runs.append([lab[0], lab[1], t, t + step])
            prev_t = t
        # absorb short runs into the previous run (or the next one at the start), then re-merge
        changed = True
        while changed and len(runs) > 1:
            changed = False
            for i, r in enumerate(runs):
                if r[3] - r[2] < min_s and r[0] != 'absent':
                    nb = [j for j in (i - 1, i + 1) if 0 <= j < len(runs) and runs[j][0] != 'absent']
                    if not nb:
                        continue                   # a short sighting between absences stays as it is
                    j = nb[0]
                    runs[j][2], runs[j][3] = min(runs[j][2], r[2]), max(runs[j][3], r[3])
                    del runs[i]; changed = True; break
            merged = [runs[0]]
            for r in runs[1:]:
                if r[0] == merged[-1][0] and r[1] == merged[-1][1]:
                    merged[-1][3] = max(merged[-1][3], r[3])
                else:
                    merged.append(r)
            runs = merged
        if runs[0][2] > fill:
            runs.insert(0, ['absent', None, 0.0, runs[0][2]])
        if duration - runs[-1][3] > fill:
            runs.append(['absent', None, runs[-1][3], duration])
        ivs = [{'state': s, 'target': tg, 't0': round(a, 2), 't1': round(min(b, duration), 2)} for s, tg, a, b in runs]
        objects[oid] = {'cls': summ[oid]['cls'], 'src': summ[oid]['src'], 'intervals': ivs}
        for prev, cur in zip(ivs, ivs[1:]):
            changes.append({'obj': oid, 't': cur['t0'], 'from': [prev['state'], prev['target']],
                            'to': [cur['state'], cur['target']]})
    changes.sort(key=lambda c: (c['t'], c['obj']))
    for i, c in enumerate(changes, 1):
        c['id'] = f'C{i}'
    return objects, changes

## 6. Run

Each video is extracted from its zip, processed, saved (three files), and deleted from the Colab disk. The states file is written last and marks the video as done.

In [61]:
SRC = VideoSource()
RUN = Run(params={'step': 'cv_detect_track', 'shard': SHARD, 'n_shards': N_SHARDS, 'groups': GROUPS,
                  'limit': LIMIT_PER_GROUP, 'det': DET_NAME, 'openvocab': OV_NAME, 'stitch': EMB['method']},
          items_total=len(MY))
TIMES = []

def process(row):
    t0 = time.time()
    path = SRC.get(row)
    pv = probe_video(path)
    frames, ov_frames, fc, lc, shape = scan_video(path)
    duration = pv['duration_s'] or (frames[-1]['t'] + 1.0 / CVC['decode_fps'])
    out_frames, summ, links = build_tracks(frames, ov_frames, fc, lc)
    objects, changes = build_states(out_frames, summ, duration)
    models = {'det': DET_NAME, 'openvocab': OV_NAME, 'tracker': CVC['tracker'], 'stitch': EMB['method'],
              'stitch_threshold': EMB['threshold']}
    save_compact({'key': row.key, 'group': row.group, 'decode_fps': CVC['decode_fps'], 'src_fps': pv['fps'],
                  'width': int(shape[1]), 'height': int(shape[0]), 'duration_s': round(float(duration), 3),
                  'times': [f['t'] for f in frames]}, CVD['frames_index'] / f'{row.key}.json')
    save_compact({'key': row.key, 'models': models, 'tracks': summ, 'stitch_links': links, 'frames': out_frames},
                 CVD['tracks'] / f'{row.key}.json')
    save_compact({'key': row.key, 'duration_s': round(float(duration), 3), 'objects': objects, 'changes': changes,
                  'rules': {k: CVC[k] for k in ['state_fill_gap_s', 'state_min_s']}},
                 CVD['states'] / f'{row.key}.json')
    SRC.drop(row)
    TIMES.append({'key': row.key, 'seconds': round(time.time() - t0, 1), 'video_s': round(float(duration), 1),
                  'tracks': len(summ), 'stitched': len(links), 'changes': len(changes)})

SUBDIRS_02 = ['frames_index', 'tracks', 'states']
N_SINCE_SYNC = [0]
def process_and_sync(row):
    process(row)
    N_SINCE_SYNC[0] += 1
    if N_SINCE_SYNC[0] >= SYNC_EVERY:
        sync_to_drive(SUBDIRS_02, '02')
        N_SINCE_SYNC[0] = 0

todo = list(MY.itertuples(index=False))
try:
    res = run_resumable(todo, key=lambda r: r.key, fn=process_and_sync, done=done_ids(CVD['states']),
                        desc='CV videos', progress_every=10)
    print(res)
finally:
    sync_to_drive(SUBDIRS_02, '02')            # always upload what is done, even after an error

[14:14:39] INFO  run started: 02_cv_detect_track-20261016-141438-207a17
[14:14:39] INFO  CV videos: 851 total, 47 already done, 804 to do
[14:16:08] INFO  CV videos: 10/804 | 8.9s each | ETA 117.9 min
[14:17:51] INFO  CV videos: 20/804 | 9.6s each | ETA 125.5 min
[14:18:46] INFO  uploaded 02_shard0_20261016-141846_00075.zip: 75 files (0.0 MB), verified on Drive
[14:19:31] INFO  CV videos: 30/804 | 9.8s each | ETA 125.9 min
[14:21:10] INFO  CV videos: 40/804 | 9.8s each | ETA 124.7 min
[14:21:54] INFO  uploaded 02_shard0_20261016-142154_00075.zip: 75 files (0.0 MB), verified on Drive
[14:21:54] INFO  CV videos: 50/804 | 8.7s each | ETA 109.4 min
[14:22:07] INFO  CV videos: 60/804 | 7.5s each | ETA 92.6 min
[14:22:20] INFO  CV videos: 70/804 | 6.6s each | ETA 80.6 min
[14:22:25] INFO  uploaded 02_shard0_20261016-142225_00075.zip: 75 files (0.0 MB), verified on Drive
[14:22:31] INFO  CV videos: 80/804 | 5.9s each | ETA 71.2 min
[14:22:43] INFO  CV videos: 90/804 | 5.4s each | ETA 64.1 min

## 7. Check and record

Prints this member's progress and a few sanity numbers, then closes the run. If `remaining` above is not 0, run the cell above again first.

In [62]:
sync_to_drive(SUBDIRS_02, '02')                # safe to run again any time
done_now = {Path(r).stem for r in _PACKED if r.startswith('states/')}   # done = safely on Drive
prog = MY.assign(done=MY.key.isin(done_now)).groupby('group', sort=False).done.agg(['sum', 'size'])
print(prog.rename(columns={'sum': 'done (on Drive)', 'size': 'videos'}))
print(f'packs on Drive: {len(list(PACKS.glob("02_*.zip")))}')
if TIMES:
    T = pd.DataFrame(TIMES)
    print(f"\nthis session: {len(T)} videos, {T.seconds.sum() / 60:.1f} min, "
          f"{T.seconds.sum() / max(T.video_s.sum(), 1e-6):.2f} s of compute per video second")
    print(f"per video: median {T.tracks.median():.0f} tracks, {T.stitched.median():.0f} stitched joins, "
          f"{T.changes.median():.0f} candidate state changes")
# one example, to eyeball
ex = next(iter(sorted(done_now & set(MY.key))), None)
if ex:
    st = load_json(CVD['states'] / f'{ex}.json')
    print(f'\nexample {ex}: {len(st["objects"])} objects')
    for oid, o in list(st['objects'].items())[:4]:
        print(f'  {oid:18s}', ' | '.join(f"{i['t0']:.1f}-{i['t1']:.1f}s {i['state']}{' ' + i['target'] if i['target'] else ''}"
                                        for i in o['intervals'][:5]))
left = int((~MY.key.isin(done_now)).sum())
save_json({'shard': SHARD, 'videos': len(MY), 'done': len(MY) - left, 'updated': dt.datetime.now().isoformat(timespec='seconds')},
          P['cv'] / f'progress_02_shard{SHARD}.json')
RUN.finish('done' if left == 0 else 'partial', notes=f'{left} videos left')

          done (on Drive)  videos
group                            
dev                    92      92
star                  360     360
main                  175     175
intentqa              224     224
packs on Drive: 34

this session: 804 videos, 85.5 min, 0.21 s of compute per video second
per video: median 27 tracks, 2 stitched joins, 23 candidate state changes

example 10008553263: 9 objects
  speaker_ov8        0.0-2.0s absent | 2.0-3.0s near person_1 | 3.0-4.0s absent | 4.0-5.0s near person_1 | 5.0-14.1s absent
  microphone_ov16    0.0-5.0s absent | 5.0-10.0s held_by person_1 | 10.0-14.1s absent
  wall_ov17          0.0-5.0s absent | 5.0-7.0s visible | 7.0-8.0s absent | 8.0-9.0s visible | 9.0-14.1s absent
  drum_ov27          0.0-12.0s absent | 12.0-14.0s held_by person_25 | 14.0-14.1s near person_25
[15:44:05] INFO  run finished: 02_cv_detect_track-20261016-141438-207a17 [done] 89.4 min, peak RAM 2.6 GB, peak VRAM 1.6 GB


## What to send back

When `done` equals `videos` for every group, send the output of section 7. Then start `03_cv_events` with the same `SHARD`. Member B's videos do not need to be finished first: 03 only needs the videos of its own shard.